# Supplementary analyses (thesis Section 5.5)

Five analyses computed **from the saved outputs of the main run** (`Honours_Every_Third_MI_AIS_TE_v5.ipynb`),
without re-running it and without modifying anything it wrote. Run this notebook from the same `notebooks/`
directory, with the same kernel, after the main run has finished. Outputs go to `<run>/thesis_v5/supplementary/`.

| # | Analysis | Thesis hypothesis / section | Inputs |
|---|---|---|---|
| S1 | Stride-1 source-delay sweep extended to delays 4–12 simulator rows | H2, §5.2 | raw recording, `groups.npz`, `te.npz` |
| S2 | Direction accuracy by edge class (pull-down vs pass) | H3/H4, §5.2 | `te.npz`, netlist |
| S3 | Pairwise TE against interventional causal effect, pair by pair | H7, §5.4 | `simulated_chip/states.npz`, `interventions.npz` |
| S4 | Case study: one named functional unit — wiring vs pairwise TE vs IDTxl | §5.3/5.4 | netlist names, `te.npz`, `idtxl/network_*.npz` |
| S5 | RQ2 networks as ROC / precision–recall operating points, with partial-ranking curves | H5, §5.3 | `te.npz`, `idtxl/network_*.npz` |

Environment overrides (optional): `TE6502_ROOT` (project root), `TE6502_OUT` (run directory),
`TE6502_RQ_DIR` (results sub-directory, default `thesis_v5`), `TE6502_CASE_MODULE` (module name for S4).

In [ ]:
import os, sys, json, time, ast, math, re, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import spearmanr
try:
    from IPython.display import display, Markdown
except Exception:  # plain-python execution
    display = print
    Markdown = str

def _find_root():
    env = os.environ.get('TE6502_ROOT')
    if env:
        return Path(env)
    return next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data/raw/donkeykong.5000.ts.spikes.npy').exists())

ROOT = _find_root()
DATA_PATH = ROOT / 'data/raw/donkeykong.5000.ts.spikes.npy'
NET_DIR = ROOT / 'data/visual6502'
OUT = Path(os.environ['TE6502_OUT']) if os.environ.get('TE6502_OUT') else ROOT / 'results/v5_bitcorrect_every_third_mi_ais_te_k1_12_L1_d1_ais_x1'
RQ_DIR = OUT / os.environ.get('TE6502_RQ_DIR', 'thesis_v5')
SIM_DIR = RQ_DIR / 'simulated_chip'; IDX_DIR = RQ_DIR / 'idtxl'
SUP_DIR = RQ_DIR / 'supplementary'; SUP_FIG = SUP_DIR / 'figures'; SUP_TAB = SUP_DIR / 'tables'
for _d in (SUP_DIR, SUP_FIG, SUP_TAB):
    _d.mkdir(parents=True, exist_ok=True)
for _p in (Path.cwd(), ROOT / 'notebooks'):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
import te6502_toolkit as tk
import sim6502
importlib.reload(tk); importlib.reload(sim6502)

RQ = json.loads((RQ_DIR / 'configuration.json').read_text())
SIM = json.loads((SIM_DIR / 'configuration.json').read_text()) if (SIM_DIR / 'configuration.json').exists() else None
if (RQ_DIR / 'run_health.json').exists():
    _h = pd.DataFrame(json.loads((RQ_DIR / 'run_health.json').read_text()))
    print('Main-run health report:', int((_h.status == 'PASS').sum()), 'PASS /', int((_h.status != 'PASS').sum()), 'not PASS')
    if (_h.status != 'PASS').any():
        print(_h[_h.status != 'PASS'].to_string())
else:
    print('WARNING: no run_health.json in', RQ_DIR, '- the main run has not reached its final section.')
print('Run directory:', OUT); print('Results:', RQ_DIR); print('Supplementary outputs:', SUP_DIR)

plt.rcParams.update({'savefig.dpi': 300, 'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.titlesize': 9, 'legend.fontsize': 8, 'figure.dpi': 110})

def sup_json(value):
    if isinstance(value, dict): return {str(k): sup_json(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [sup_json(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating, float)): return None if not np.isfinite(value) else float(value)
    if isinstance(value, np.bool_): return bool(value)
    if isinstance(value, np.ndarray): return sup_json(value.tolist())
    return value

LEDGER_PATH = SUP_DIR / 'supplementary_ledger.json'
LEDGER = json.loads(LEDGER_PATH.read_text()) if LEDGER_PATH.exists() else {}
FIGURES, TABLES = {}, {}
def sup_record(key, value):
    LEDGER[key] = sup_json(value); LEDGER_PATH.write_text(json.dumps(LEDGER, indent=2))
def sup_fig(fig, name, caption):
    for ext in ('pdf', 'png'):
        fig.savefig(SUP_FIG / f'{name}.{ext}', bbox_inches='tight')
    FIGURES[name] = caption; plt.show(); plt.close(fig)
def sup_table(df, name, caption, show=True, fmt='{:.4g}'):
    df.to_csv(SUP_TAB / f'{name}.csv', index=False)
    tk.latex_table(df, SUP_TAB / f'{name}.tex', caption, f'tab:{name}', fmt)
    TABLES[name] = caption
    if show: display(df)
def rq_ci(point, lo, hi, digits=3):
    f = f'{{:.{digits}f}}'
    return '--' if not np.isfinite(point) else f'{f.format(point)} [{f.format(lo)}, {f.format(hi)}]'
def main_table(name):
    p = RQ_DIR / 'tables' / f'{name}.csv'
    return pd.read_csv(p) if p.exists() else None

## 0. Load the run and rebuild the structural reference
This is the main notebook's shared setup (its Section 5), reproduced so that the supplementary analyses use
exactly the same activity groups, netlist relation and evaluation weights. The equality check against
`analysis_pairwise/structural_counts.npz` guarantees the reference is identical.

In [ ]:
with np.load(OUT / 'groups.npz') as z:
    rq_reps = z['representatives'].astype(int); rq_p2g = z['physical_to_group'].astype(int)
with np.load(OUT / 'reduction_audit.npz') as z:
    rq_phase = int(z['phase']); rq_stride = int(z['stride']); rq_stop = int(z['original_active_rows'])
with np.load(OUT / 'te.npz') as z:
    RQ_TE = z['te_bits'].copy(); RQ_LOGP = z['log_p'].copy()
    RQ_LOGP = np.where(np.isneginf(RQ_LOGP), np.nanmin(np.where(np.isfinite(RQ_LOGP), RQ_LOGP, 0)) - 1e3, RQ_LOGP)
    RQ_K = z['optimal_k'].astype(int)
with np.load(OUT / 'mi.npz') as z:
    RQ_MI = z['mi_bits'].copy()
RQ_TE0 = None
if (RQ_DIR / 'te_lag0.npz').exists():
    with np.load(RQ_DIR / 'te_lag0.npz') as z:
        RQ_TE0 = z['te_bits'].copy()
rq_raw = np.load(DATA_PATH, mmap_mode='r')

def rq_series(phase=None, stride=None, block=40000):
    """[group, time] uint8 series for any phase/stride using this run's representatives."""
    phase = rq_phase if phase is None else int(phase); stride = rq_stride if stride is None else int(stride)
    rows = np.arange(phase, rq_stop, stride)
    out = np.empty((len(rq_reps), len(rows)), dtype=np.uint8)
    for a in range(0, len(rows), block):
        r = rows[a:a + block]
        out[:, a:a + len(r)] = np.asarray(rq_raw[r[0]:r[-1] + 1:stride])[:, rq_reps].T
    return out

RQ_X = rq_series()
RQ_G, RQ_N = RQ_X.shape
RQ_RATE = RQ_X.mean(axis=1)
assert RQ_TE.shape == (RQ_G, RQ_G) and len(RQ_K) == RQ_G

VSS, VCC = 558, 657
rq_defs = []
for line in (NET_DIR / 'transdefs.js').read_text().splitlines():
    if line.strip().startswith("['t"):
        rq_defs.append(ast.literal_eval(line.strip().rstrip(',')))
rq_defs.sort(key=lambda d: int(d[0][1:]))
RQ_M = len(rq_defs)
assert RQ_M == len(rq_p2g), 'Netlist and recording disagree on transistor count.'
rq_gate, rq_c1, rq_c2 = (np.asarray([d[i] for d in rq_defs]) for i in (1, 2, 3))
rq_box = np.asarray([d[4] for d in rq_defs], float)
RQ_CENTRE = np.column_stack(((rq_box[:, 0] + rq_box[:, 1]) / 2, (rq_box[:, 2] + rq_box[:, 3]) / 2))
_gate_lookup = {}
for t_, g_ in enumerate(rq_gate):
    _gate_lookup.setdefault(int(g_), []).append(t_)
_src, _dst, _pd = [], [], []
for s_ in range(RQ_M):
    dests = set()
    for term in (rq_c1[s_], rq_c2[s_]):
        if term not in (VSS, VCC):
            dests.update(_gate_lookup.get(int(term), []))
    dests.discard(s_)
    for t_ in dests:
        _src.append(s_); _dst.append(t_); _pd.append(VSS in (rq_c1[s_], rq_c2[s_]))
_src, _dst, _pd = map(np.asarray, (_src, _dst, _pd))
def _csr(mask=None):
    s, d = (_src, _dst) if mask is None else (_src[mask], _dst[mask])
    return sparse.csr_matrix((np.ones(len(s)), (s, d)), shape=(RQ_M, RQ_M))
RQ_A = _csr(); RQ_A_PD = _csr(_pd); RQ_A_PASS = _csr(~_pd)
_active = np.flatnonzero(rq_p2g >= 0)
RQ_P = sparse.csr_matrix((np.ones(len(_active)), (_active, rq_p2g[_active])), shape=(RQ_M, RQ_G))
def rq_agg(B):
    C = np.asarray((RQ_P.T @ sparse.csr_matrix(B) @ RQ_P).todense()); np.fill_diagonal(C, 0); return C
RQ_POS = rq_agg(RQ_A); RQ_POS_PD = rq_agg(RQ_A_PD); RQ_POS_PASS = rq_agg(RQ_A_PASS)
RQ_SIZE = np.bincount(rq_p2g[_active], minlength=RQ_G).astype(float)
RQ_PAIRS = np.outer(RQ_SIZE, RQ_SIZE); np.fill_diagonal(RQ_PAIRS, 0)
RQ_NEG = RQ_PAIRS - RQ_POS
assert (RQ_NEG >= 0).all()
legacy_counts = OUT / 'analysis_pairwise/structural_counts.npz'
if legacy_counts.exists():
    with np.load(legacy_counts) as z:
        assert np.array_equal(z['positive_count'], RQ_POS), 'Structural reference differs from the main run.'
    print('Structural reference identical to the main run: OK')
RQ_EL = ~np.eye(RQ_G, dtype=bool)
RQ_COL = np.broadcast_to(np.arange(RQ_G)[None, :], (RQ_G, RQ_G))[RQ_EL]
def rq_weights(level):
    if level == 'physical':
        return RQ_POS[RQ_EL].astype(float), RQ_NEG[RQ_EL].astype(float)
    return (RQ_POS[RQ_EL] > 0).astype(float), (RQ_POS[RQ_EL] == 0).astype(float)

class Ranked:
    """Pre-sorted score for fast cluster-bootstrap metrics (weights change, order does not)."""
    def __init__(self, score_matrix, level, mask=None):
        s = np.asarray(score_matrix, float)[RQ_EL]; p, n = rq_weights(level); c = RQ_COL
        keep = np.isfinite(s) & ((p + n) > 0) & (True if mask is None else mask[RQ_EL])
        order = np.argsort(-s[keep], kind='stable')
        self.s, self.p, self.n, self.c = s[keep][order], p[keep][order], n[keep][order], c[keep][order]
        self.ends = np.r_[np.flatnonzero(np.diff(self.s) != 0), len(self.s) - 1]
    def metrics(self, mult=None, budgets=()):
        w = np.ones(len(self.c)) if mult is None else mult[self.c]
        p, n = self.p * w, self.n * w
        tp = np.cumsum(p)[self.ends]; fp = np.cumsum(n)[self.ends]; items = np.cumsum(w)[self.ends]
        P, Nn = tp[-1], fp[-1]
        if P <= 0 or Nn <= 0: raise ValueError('degenerate draw')
        tpr = np.r_[0, tp / P]; fpr = np.r_[0, fp / Nn]
        prec = tp / np.maximum(tp + fp, 1e-300)
        out = dict(AUC=tk._trapz(tpr, fpr), AP=float(np.sum(np.diff(np.r_[0, tp / P]) * prec)), prevalence=P / (P + Nn))
        out['AP_lift'] = out['AP'] / out['prevalence']
        for b in budgets:
            i = int(np.searchsorted(items, b))
            if i >= len(items): i = len(items) - 1
            prev_items = items[i - 1] if i > 0 else 0.0; prev_tp = tp[i - 1] if i > 0 else 0.0; prev_fp = fp[i - 1] if i > 0 else 0.0
            frac = (b - prev_items) / max(items[i] - prev_items, 1e-300); frac = min(max(frac, 0), 1)
            t_ = prev_tp + frac * (tp[i] - prev_tp); f_ = prev_fp + frac * (fp[i] - prev_fp)
            out[f'P@{b}'] = t_ / (t_ + f_) if t_ + f_ > 0 else np.nan
            out[f'lift@{b}'] = out[f'P@{b}'] / out['prevalence']
        return out
    def curves(self):
        tp = np.cumsum(self.p)[self.ends]; fp = np.cumsum(self.n)[self.ends]
        return dict(fpr=np.r_[0, fp / fp[-1]], tpr=np.r_[0, tp / tp[-1]], recall=tp / tp[-1], precision=tp / (tp + fp))

print(f'{RQ_G} activity groups, {RQ_N:,} samples, {int(RQ_POS.sum()):,} evaluable wired transistor pairs, '
      f'{int((RQ_POS > 0).sum()):,} wired group pairs')

## S1. Stride-1 source-delay sweep, delays 4–12 simulator rows
The main run evaluated pairwise TE on the unreduced (stride-1) recording at source delays of 1, 2, 3 and 6 rows
with a common history $k=3$. Three rows make one clock phase, so delays 1–2 are *within* the phase of the target
sample, 3 is one phase back, 6 two phases back. This sweep adds delays 4–12 (delay 6 is recomputed as a check
against the main run) to show whether structural discrimination is periodic in the phase and where it peaks.

In [ ]:
SUPP_DELAYS = [4, 5, 6, 7, 8, 9, 10, 11, 12]
S1_PATH = SUP_DIR / 'stride1_sweep.json'
sweep_main = main_table('rq1_phase_delay_sweep')
def sweep_eval(S, label):
    out = dict(label)
    for level in ['physical', 'group']:
        m = Ranked(np.where(np.isfinite(S), S, 0.0), level).metrics(budgets=[1000])
        out[f'{level}_AUC'] = m['AUC']; out[f'{level}_AP_lift'] = m['AP_lift']; out[f'{level}_lift@1000'] = m['lift@1000']
    return out
if S1_PATH.exists():
    s1_rows = json.loads(S1_PATH.read_text())
else:
    X1 = rq_series(phase=0, stride=1); s1_rows = []
    const1 = float(np.mean(X1.min(1) == X1.max(1)))
    for d in SUPP_DELAYS:
        t0 = time.perf_counter(); S, _ = tk.te_matrix(X1, np.full(RQ_G, RQ['stride1_k']), delay=d)
        s1_rows.append(sweep_eval(S, dict(stride=1, phase=0, delay_samples=d, delay_rows=d, constant_group_fraction=const1,
                                          seconds=time.perf_counter() - t0, source='supplementary')))
        print(f'  stride 1 delay {d}: {time.perf_counter() - t0:.0f}s', flush=True)
    del X1
    S1_PATH.write_text(json.dumps(sup_json(s1_rows), indent=1))
S1 = pd.DataFrame(s1_rows)
if sweep_main is not None:
    m1 = sweep_main[(sweep_main.stride == 1)].copy(); m1['source'] = 'main run'
    chk = pd.merge(m1[m1.delay_rows == 6], S1[S1.delay_rows == 6], on='delay_rows', suffixes=('_main', '_supp'))
    if len(chk):
        diff = float(abs(chk.physical_AUC_main.iloc[0] - chk.physical_AUC_supp.iloc[0]))
        print(f'Check: delay-6 physical AUC main {chk.physical_AUC_main.iloc[0]:.4f} vs supplementary {chk.physical_AUC_supp.iloc[0]:.4f} (|diff| {diff:.1e})')
        sup_record('s1_delay6_check', dict(main=float(chk.physical_AUC_main.iloc[0]), supplementary=float(chk.physical_AUC_supp.iloc[0]), abs_diff=diff))
    ALL1 = pd.concat([m1[m1.delay_rows != 6], S1], ignore_index=True).sort_values('delay_rows')
    ref3 = sweep_main[(sweep_main.stride == 3) & (sweep_main.phase == rq_phase)]
else:
    ALL1 = S1.sort_values('delay_rows'); ref3 = None
cols = ['source', 'delay_rows', 'physical_AUC', 'physical_AP_lift', 'physical_lift@1000', 'group_AUC', 'group_AP_lift', 'group_lift@1000', 'seconds']
sup_table(ALL1[cols], 'supp_stride1_delays', 'Stride-1 (unreduced) recording: pairwise TE with k = 3 at source delays of 1-12 simulator rows (three rows = one clock phase). Delays 1, 2, 3 from the main run; 4-12 from this supplement.')
best = ALL1.loc[ALL1.physical_AUC.idxmax()]
sup_record('s1_stride1_sweep', dict(rows=ALL1[cols].to_dict('records'), best_delay_rows=int(best.delay_rows), best_physical_AUC=float(best.physical_AUC)))
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), constrained_layout=True)
for ax, col, base, yl in [(axes[0], 'physical_AUC', .5, 'Physical ROC-AUC'), (axes[1], 'group_AP_lift', 1, 'Group AP lift')]:
    for d in range(3, 13, 3):
        ax.axvline(d, c='0.85', lw=5, zorder=0)
    ax.plot(ALL1.delay_rows, ALL1[col], marker='o', c='C0', label='stride 1, k = 3')
    if ref3 is not None and len(ref3):
        ax.plot(ref3.delay_rows, ref3[col], marker='s', ls='none', c='C3', label=f'stride 3, phase {rq_phase}, AIS k (main run)')
    ax.axhline(base, c='grey', ls=':'); ax.set(xlabel='Source delay (simulator rows; shaded = whole clock phases)', ylabel=yl, xticks=range(0, 13))
axes[1].legend(fontsize=7)
sup_fig(fig, 'figS1_stride1_delays', 'Structural discrimination of pairwise TE on the unreduced recording as a function of source delay in simulator rows; shaded bands mark delays of whole clock phases.')
print('Best stride-1 delay by physical AUC:', int(best.delay_rows), 'rows; AUC', round(float(best.physical_AUC), 3))

## S2. Direction accuracy by edge class
The main run's H3 analysis scores the orientation of every *unidirectionally* wired group pair, weighting each
pair by the number of physical edges it stands for. Here the same analysis is split by edge class: physical
pull-down edges (source discharges the node that gates the target; inverting) and pass edges (source transmits a
value). If the rising-edge representation hides inverting influence (mechanism 2), TE asymmetry should orient
pass edges better than pull-down edges, while nuisance predictors should not differ.

In [ ]:
uni = (RQ_POS > 0) & (RQ_POS.T == 0) & RQ_EL
I_, J_ = np.nonzero(uni)
W_CLASS = {'all edges': RQ_POS[I_, J_].astype(float), 'pull-down edges': RQ_POS_PD[I_, J_].astype(float), 'pass edges': RQ_POS_PASS[I_, J_].astype(float)}
t0 = time.perf_counter(); TE_K4 = tk.te_matrix(RQ_X, np.full(RQ_G, 4))[0]; print(f'  common-k TE (k = 4) in {time.perf_counter() - t0:.0f}s')
RQ_HCOND = np.asarray([tk.conditional_entropy_bits(RQ_X[g], int(RQ_K[g])) for g in range(RQ_G)])
H4 = np.asarray([tk.conditional_entropy_bits(RQ_X[g], 4) for g in range(RQ_G)])
dir_scores = {'TE asymmetry, saved target-specific k': RQ_TE, 'TE asymmetry, common k = 4': TE_K4}
if RQ_TE0 is not None:
    dir_scores['Lag-0 conditional MI asymmetry, saved k'] = RQ_TE0
nuisance = {'Nuisance: higher event rate is target': RQ_RATE, 'Nuisance: higher H(Y|Y^(4)) is target': H4, 'Nuisance: higher H(Y|Y^(k)) is target (saved k)': RQ_HCOND}

def oriented_accuracy(fwd, rev, w, mult=None):
    ww = w if mult is None else w * mult
    correct = (fwd > rev); tie = np.isclose(fwd, rev, rtol=0, atol=1e-15)
    return float((ww * (correct & ~tie)).sum() / max((ww * ~tie).sum(), 1e-300)), float((ww * tie).sum() / max(ww.sum(), 1e-300))

rows, contrast_rows = [], []
for name, S in list(dir_scores.items()) + list(nuisance.items()):
    is_te = name in dir_scores
    fwd, rev = (S[I_, J_], S[J_, I_]) if is_te else (S[J_], S[I_])
    draws_by_class = {}
    for cls, w in W_CLASS.items():
        acc, ties = oriented_accuracy(fwd, rev, w)
        draws = tk.bootstrap_targets(lambda m, w=w: dict(a=oriented_accuracy(fwd, rev, w, m[J_])[0]), RQ_G, RQ['bootstrap_draws'], RQ['seed'])
        draws_by_class[cls] = draws
        lo, hi, _ = tk.interval(draws, 'a')
        rows.append(dict(predictor=name, edge_class=cls, physical_edges=float(w.sum()), accuracy=acc, ci_low=lo, ci_high=hi, accuracy_ci=rq_ci(acc, lo, hi), tie_fraction=ties))
    # paired contrast pass minus pull-down (same bootstrap draws: same seed -> same multiplicities)
    a_pass = {d['draw']: d['a'] for d in draws_by_class['pass edges']}; a_pd = {d['draw']: d['a'] for d in draws_by_class['pull-down edges']}
    common = sorted(set(a_pass) & set(a_pd)); diffs = np.asarray([a_pass[d] - a_pd[d] for d in common])
    point = oriented_accuracy(fwd, rev, W_CLASS['pass edges'])[0] - oriented_accuracy(fwd, rev, W_CLASS['pull-down edges'])[0]
    contrast_rows.append(dict(predictor=name, contrast='pass minus pull-down', difference=point, ci_low=float(np.quantile(diffs, .025)) if len(diffs) else np.nan,
                              ci_high=float(np.quantile(diffs, .975)) if len(diffs) else np.nan, fraction_draws_positive=float(np.mean(diffs > 0)) if len(diffs) else np.nan))
DIRC = pd.DataFrame(rows); DIRD = pd.DataFrame(contrast_rows)
sup_table(DIRC[['predictor', 'edge_class', 'physical_edges', 'accuracy_ci', 'tie_fraction']], 'supp_direction_by_class',
          'Orientation accuracy on unidirectionally wired group pairs, weighted by physical edges of each class, with 95% target-cluster bootstrap intervals.')
sup_table(DIRD, 'supp_direction_class_contrast', 'Paired bootstrap contrast of orientation accuracy: pass edges minus pull-down edges, per predictor.')
sup_record('s2_direction_by_class', dict(table=DIRC.drop(columns='accuracy_ci').to_dict('records'), contrasts=DIRD.to_dict('records'), unidirectional_group_pairs=int(len(I_))))
fig, ax = plt.subplots(figsize=(7, 3.8), constrained_layout=True)
preds = list(DIRC.predictor.unique()); classes = list(W_CLASS); h = .26
for ci, cls in enumerate(classes):
    d = DIRC[DIRC.edge_class == cls].set_index('predictor').loc[preds]
    y = np.arange(len(preds)) + (ci - 1) * h
    ax.barh(y, d.accuracy, height=h, xerr=[d.accuracy - d.ci_low, d.ci_high - d.accuracy], label=cls, color=f'C{ci}', error_kw=dict(lw=.8))
ax.set_yticks(np.arange(len(preds)), preds); ax.axvline(.5, c='k', ls='--', lw=1); ax.set(xlabel='Physical-edge-weighted orientation accuracy', xlim=(0, 1)); ax.invert_yaxis(); ax.legend(fontsize=7, loc='lower right')
sup_fig(fig, 'figS2_direction_by_class', 'Orientation accuracy of TE asymmetry and nuisance predictors, by edge class (all, pull-down, pass), with 95% bootstrap intervals.')

## S3. Pairwise TE against interventional causal effect on the simulated chip
The main run reports, for the simulated chip, how well TE *ranks* causal pairs (ROC-AUC against the clamping
references). This supplement looks at the relation pair by pair: for every intervened source node and every
target node, the measured causal effect ($E_0$: same half-step, $E_1$: next half-step; fraction of moments at
which clamping the source changed the target) against pairwise TE at the matching lag (state representation,
AIS-selected histories). It quantifies the two disagreements the theory predicts (Lizier & Prokopenko 2010):
causal effect without transfer, and transfer without causal effect.

In [ ]:
S3_OK = (SIM_DIR / 'states.npz').exists() and (SIM_DIR / 'interventions.npz').exists()
if not S3_OK:
    print('S3 skipped: simulated_chip/states.npz or interventions.npz not found (main run has not completed Part V).')
else:
    with np.load(SIM_DIR / 'states.npz') as z:
        S_STATE = z['states']; S_NODES = np.asarray(z['nodes'])
    with np.load(SIM_DIR / 'interventions.npz') as z:
        E0, E1, int_nodes, n_times = z['E0'], z['E1'], z['nodes'].astype(int), int(z['n_times'])
    CHIP = sim6502.Chip(NET_DIR)
    g_, c1_, c2_ = CHIP.transistor_matrix()
    S_A = tk.node_reference_from_transistors(g_, c1_, c2_, S_NODES, supply={CHIP.gnd, CHIP.pwr})
    A_dense = S_A.toarray() > 0
    X_state = S_STATE.T.astype(np.uint8)
    reps_s, ug_s = tk.group_identical(X_state); Xs = X_state[reps_s]; Gs = len(reps_s)
    S3_TE = SUP_DIR / 'sim_state_te.npz'
    if S3_TE.exists():
        with np.load(S3_TE) as z: TE_s0, TE_s1, ks_state = z['te0'], z['te1'], z['k']
    else:
        t0 = time.perf_counter(); ks_state = np.asarray([tk.ais_select_k(x, SIM['k_max']) for x in Xs])
        TE_s1, _ = tk.te_matrix(Xs, ks_state, delay=1); TE_s0, _ = tk.te_matrix(Xs, ks_state, delay=0)
        np.savez_compressed(S3_TE, te0=TE_s0, te1=TE_s1, k=ks_state); print(f'  simulated-chip TE (lags 0, 1) on {Gs} groups in {time.perf_counter() - t0:.0f}s')
    # Pair table: intervened source node i, target node j (both in non-constant groups, different groups).
    nU = len(S_NODES); src_g = ug_s[int_nodes]
    I2 = np.repeat(np.arange(len(int_nodes)), nU); J2 = np.tile(np.arange(nU), len(int_nodes))
    gi, gj = src_g[I2], ug_s[J2]
    ok = (gi >= 0) & (gj >= 0) & (gi != gj)
    I2, J2, gi, gj = I2[ok], J2[ok], gi[ok], gj[ok]
    PAIRS = pd.DataFrame(dict(source=S_NODES[int_nodes[I2]], target=S_NODES[J2], E0=E0[I2, J2].astype(float), E1=E1[I2, J2].astype(float),
                              TE0=TE_s0[gi, gj], TE1=TE_s1[gi, gj], wired=A_dense[int_nodes[I2], J2]))
    PAIRS = PAIRS[np.isfinite(PAIRS.TE0) & np.isfinite(PAIRS.TE1)].reset_index(drop=True)
    rows_ = []
    for lag, (ec, tc) in {0: ('E0', 'TE0'), 1: ('E1', 'TE1')}.items():
        e, t = PAIRS[ec].to_numpy(), PAIRS[tc].to_numpy(); w = PAIRS.wired.to_numpy()
        null_ref = t[(e == 0) & ~w]; tau = float(np.quantile(null_ref, .95)) if len(null_ref) else np.nan
        causal = e > 0; transfer = t > tau
        rho_all = spearmanr(e, t); rho_c = spearmanr(e[causal], t[causal]) if causal.sum() > 10 else None
        rows_.append(dict(lag=lag, pairs=int(len(e)), causal_pairs=int(causal.sum()), wired_pairs=int(w.sum()),
                          te_threshold_bits_95pct_null=tau,
                          spearman_all=float(rho_all.statistic), spearman_causal_only=float(rho_c.statistic) if rho_c else np.nan,
                          causal_without_transfer=float((causal & ~transfer).sum() / max(causal.sum(), 1)),
                          transfer_without_causal=float((transfer & ~causal).sum() / max(transfer.sum(), 1)),
                          wired_without_causal=float((w & ~causal).sum() / max(w.sum(), 1)),
                          causal_without_wiring=float((causal & ~w).sum() / max(causal.sum(), 1)),
                          median_TE_causal=float(np.median(t[causal])) if causal.any() else np.nan,
                          median_TE_noncausal=float(np.median(t[~causal])) if (~causal).any() else np.nan))
    S3T = pd.DataFrame(rows_)
    sup_table(S3T, 'supp_te_vs_causal', 'Simulated 6502 (state representation): pairwise TE against clamping effect, pair by pair. Transfer threshold = 95th percentile of TE over unwired pairs with no causal effect.')
    sup_record('s3_te_vs_causal', S3T.to_dict('records'))
    rng = np.random.default_rng(11)
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.8), constrained_layout=True)
    for ax, lag, (ec, tc) in zip(axes, (0, 1), (('E0', 'TE0'), ('E1', 'TE1'))):
        d = PAIRS; e, t, w = d[ec].to_numpy(), d[tc].to_numpy(), d.wired.to_numpy()
        FLOOR = 1e-4; t = np.maximum(t, FLOOR)   # log axis; TE below 1e-4 bits drawn at the floor
        pos = np.flatnonzero(e > 0); neg = np.flatnonzero(e == 0)
        neg_s = rng.choice(neg, min(len(neg), 20000), replace=False); pos_s = rng.choice(pos, min(len(pos), 20000), replace=False)
        jitter = lambda n: rng.normal(0, .004, n)
        ax.scatter(e[neg_s] + jitter(len(neg_s)), t[neg_s], s=3, c='0.7', alpha=.25, label='no causal effect', rasterized=True)
        ax.scatter(e[pos_s][~w[pos_s]], t[pos_s][~w[pos_s]], s=4, c='C0', alpha=.4, label='causal, unwired', rasterized=True)
        ax.scatter(e[pos_s][w[pos_s]], t[pos_s][w[pos_s]], s=6, c='C3', alpha=.6, label='causal, wired', rasterized=True)
        bins = np.linspace(0, 1, 11); mids = []; med = []
        for a, b in zip(bins[:-1], bins[1:]):
            m = (e > a) & (e <= b)
            if m.sum() >= 20: mids.append((a + b) / 2); med.append(np.median(t[m]))
        if mids: ax.plot(mids, med, c='k', lw=1.5, marker='o', ms=3, label='median TE per effect decile')
        ax.axhline(max(S3T.loc[S3T.lag == lag, 'te_threshold_bits_95pct_null'].iloc[0], FLOOR), c='k', ls=':', lw=1)
        ax.set(xlabel=f'Causal effect $E_{lag}$ (fraction of clamping moments)', ylabel=f'Pairwise TE, lag {lag} (bits; floor $10^{{-4}}$)', yscale='log',
               ylim=(FLOOR * .7, float(np.nanmax(t)) * 2), title=f'lag {lag}: Spearman ρ = {S3T.loc[S3T.lag == lag, "spearman_all"].iloc[0]:.2f}')
    axes[0].legend(fontsize=7, loc='upper left')
    sup_fig(fig, 'figS3_te_vs_causal', 'Simulated 6502: pairwise TE against the measured causal effect of clamping, for every intervened source and target; dotted line = transfer threshold.')

## S4. Case study: one named functional unit
One functional unit of the chip, chosen as the named module with the most evaluable wired group pairs among
modules small enough to draw (override with `TE6502_CASE_MODULE`). Its activity groups are placed at their mean
die coordinates and four edge sets are overlaid: the netlist wiring among them; the top-$B$ pairwise lag-1 TE
edges and top-$B$ lag-0 edges within the unit ($B$ = number of wired pairs); and the IDTxl multivariate edges.
Green = wired, red = unwired.

In [ ]:
CHIP_NAMES = sim6502.Chip(NET_DIR).names if 'CHIP' not in globals() else CHIP.names
mod_t, src_t = tk.assign_modules(RQ_M, [(rq_gate[t], rq_c1[t], rq_c2[t]) for t in range(RQ_M)], CHIP_NAMES, RQ_A)
grp_mod = np.array(['Unassigned'] * RQ_G, dtype=object)
for g in range(RQ_G):
    m_ = mod_t[rq_p2g == g]; m_ = m_[m_ != 'Unassigned']
    if len(m_): v, c = np.unique(m_, return_counts=True); grp_mod[g] = v[np.argmax(c)]
grp_mod = grp_mod.astype(str)
cand_rows = []
for m in np.unique(grp_mod):
    if m == 'Unassigned': continue
    sel = grp_mod == m; blk = RQ_POS[np.ix_(sel, sel)]
    cand_rows.append(dict(module=m, groups=int(sel.sum()), wired_group_pairs=int((blk > 0).sum()), wired_physical_edges=int(blk.sum())))
CAND = pd.DataFrame(cand_rows).sort_values('wired_group_pairs', ascending=False)
sup_table(CAND, 'supp_case_study_candidates', 'Functional modules of the recording: activity groups and wired group pairs within each module.')
CASE = os.environ.get('TE6502_CASE_MODULE') or CAND[(CAND.groups >= 6) & (CAND.groups <= 80)].module.iloc[0]
sel = np.flatnonzero(grp_mod == CASE); nS = len(sel)
print('Case-study module:', CASE, '|', nS, 'activity groups')
centres = np.asarray([RQ_CENTRE[rq_p2g == g].mean(0) for g in sel]); sizes = RQ_SIZE[sel]
true_blk = RQ_POS[np.ix_(sel, sel)] > 0; np.fill_diagonal(true_blk, False); B = int(true_blk.sum())
def topB(S):
    s = np.asarray(S, float)[np.ix_(sel, sel)].copy(); np.fill_diagonal(s, -np.inf); s[~np.isfinite(s)] = -np.inf
    if B == 0: return np.zeros_like(true_blk)
    thr = np.sort(s.ravel())[::-1][B - 1]; return s >= thr
edge_sets = {'(a) Netlist wiring': true_blk, f'(b) Pairwise TE lag 1, top-{B}': topB(RQ_TE)}
if RQ_TE0 is not None: edge_sets[f'(c) Lag-0 conditional MI, top-{B}'] = topB(RQ_TE0)
NETS_IDX = {}
for method in ['multivariate', 'bivariate']:
    p = IDX_DIR / f'network_{method}.npz'
    if p.exists():
        with np.load(p) as z: NETS_IDX[method] = dict(A=z['adjacency'].astype(bool), W=z['selected_te'], targets=z['targets'].astype(int))
if 'multivariate' in NETS_IDX:
    Ai = NETS_IDX['multivariate']['A'][np.ix_(sel, sel)].copy(); np.fill_diagonal(Ai, False)
    edge_sets['(d) IDTxl multivariate TE'] = Ai
rows_ = []
for name, Eset in edge_sets.items():
    tp = int((Eset & true_blk).sum()); n = int(Eset.sum())
    rows_.append(dict(panel=name, edges=n, true_positive=tp, precision=tp / n if n else np.nan, recall=tp / B if B else np.nan))
CS = pd.DataFrame(rows_)
sup_table(CS, 'supp_case_study', f'Case study of the {CASE} unit ({nS} activity groups, {B} wired group pairs): edges within the unit recovered by each method.')
sup_record('s4_case_study', dict(module=CASE, groups=int(nS), wired_group_pairs=B, panels=CS.to_dict('records')))
nP = len(edge_sets); ncol = 2 if nP >= 3 else nP; nrow = int(math.ceil(nP / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.6 * ncol, 4.0 * nrow), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()
for ax in axes[nP:]: ax.axis('off')
for ax, (name, Eset) in zip(axes, edge_sets.items()):
    ax.scatter(centres[:, 0], centres[:, 1], s=12 + 2 * np.sqrt(sizes), c='k', zorder=3)
    for a_, b_ in zip(*np.nonzero(Eset)):
        col = 'green' if true_blk[a_, b_] else 'red'
        ax.annotate('', xy=centres[b_], xytext=centres[a_], arrowprops=dict(arrowstyle='->', color=col, lw=.8, alpha=.7, shrinkA=3, shrinkB=3))
    ax.set(title=name, xticks=[], yticks=[]); ax.set_aspect('equal')
fig.suptitle(f'{CASE}: {nS} activity groups at die coordinates; green = wired, red = unwired', fontsize=9)
sup_fig(fig, 'figS4_case_study', f'Case study of the {CASE} unit: netlist wiring, pairwise TE and IDTxl edges among its activity groups, drawn at die coordinates.')

## S5. RQ2 networks as operating points on the pairwise ROC and precision–recall curves
A thresholded network is one operating point; a ranking is a curve. This supplement places the four RQ2 networks
(IDTxl multivariate, IDTxl bivariate, pairwise BH, pairwise equal budget) on the ROC and precision–recall curves of
the pairwise lag-1 TE ranking restricted to the same targets, at both evaluation levels, and adds the
*partial-ranking* curves of the IDTxl networks (their edges ordered by selected conditional TE), so that the
multivariate network can be compared with the pairwise ranking at every budget up to its own size.

In [ ]:
if not NETS_IDX:
    print('S5 skipped: idtxl/network_*.npz not found.')
else:
    tg_sets = [set(json.loads(p.read_text())['target'] for p in (IDX_DIR / m).glob('target_*.json')) for m in NETS_IDX if (IDX_DIR / m).exists()]
    IDX_DONE = np.asarray(sorted(set.intersection(*tg_sets))) if tg_sets else np.arange(RQ_G)
    colmask = np.zeros((RQ_G, RQ_G), bool); colmask[:, IDX_DONE] = True; colmask &= RQ_EL
    bh_full = np.zeros((RQ_G, RQ_G), bool); bh_full[RQ_EL] = tk.bh_adjust_logp(RQ_LOGP[RQ_EL]) <= np.log(RQ['alpha'])
    nets = {}
    if 'multivariate' in NETS_IDX: nets['IDTxl multivariate TE'] = NETS_IDX['multivariate']['A'] & colmask
    if 'bivariate' in NETS_IDX: nets['IDTxl bivariate TE'] = NETS_IDX['bivariate']['A'] & colmask
    nets['Pairwise TE, BH (bit-correct)'] = bh_full & colmask
    if 'multivariate' in NETS_IDX:
        Bn = int(nets['IDTxl multivariate TE'].sum()); sc = np.where(colmask, RQ_TE, -np.inf)
        thr = np.sort(sc[colmask])[::-1][max(Bn - 1, 0)] if Bn else np.inf
        nets[f'Pairwise TE, top-{Bn} (equal budget)'] = colmask & (sc >= thr)
    cm = colmask[RQ_EL]
    op_rows, curves = [], {}
    for level in ['physical', 'group']:
        p, n = rq_weights(level); P, Nn = p[cm].sum(), n[cm].sum()
        R = Ranked(np.where(np.isfinite(RQ_TE), RQ_TE, 0.0), level, mask=colmask); c = R.curves(); m = R.metrics()
        curves[(level, 'Pairwise TE lag 1 (full ranking)')] = c
        op_rows.append(dict(level=level, method='Pairwise TE lag 1 (full ranking)', kind='curve', edges=np.nan, TPR=np.nan, FPR=np.nan, precision=np.nan, recall=np.nan, AUC=m['AUC'], AP_lift=m['AP_lift']))
        for name, A_ in nets.items():
            bm = tk.binary_metrics(A_[RQ_EL][cm], p[cm], n[cm])
            op_rows.append(dict(level=level, method=name, kind='operating point', edges=int(A_.sum()), TPR=bm['recall'], FPR=bm['FPR'], precision=bm['precision'], recall=bm['recall'], AUC=np.nan, AP_lift=np.nan))
        for method, lab in [('multivariate', 'IDTxl multivariate TE (edges ranked by conditional TE)'), ('bivariate', 'IDTxl bivariate TE (edges ranked by TE)')]:
            if method not in NETS_IDX: continue
            A_ = nets[f'IDTxl {method} TE']; W_ = NETS_IDX[method]['W']
            s = np.where(A_, W_, -np.inf)[RQ_EL][cm]; keep = np.isfinite(s); order = np.argsort(-s[keep], kind='stable')
            pp, nn = p[cm][keep][order], n[cm][keep][order]
            tp, fp = np.cumsum(pp), np.cumsum(nn)
            curves[(level, lab)] = dict(fpr=np.r_[0, fp / Nn], tpr=np.r_[0, tp / P], recall=tp / P, precision=tp / np.maximum(tp + fp, 1e-300))
            ap_partial = float(np.sum(np.diff(np.r_[0, tp / P]) * (tp / np.maximum(tp + fp, 1e-300)))) / (P / (P + Nn))
            op_rows.append(dict(level=level, method=lab, kind='partial ranking', edges=int(keep.sum()), TPR=np.nan, FPR=np.nan, precision=np.nan, recall=np.nan, AUC=np.nan, AP_lift=ap_partial))
    OP = pd.DataFrame(op_rows)
    sup_table(OP, 'supp_rq2_operating_points', f'RQ2 networks on the {len(IDX_DONE)} analysed targets as operating points (TPR, FPR, precision, recall) against the pairwise lag-1 TE ranking; AP lift of the IDTxl partial rankings is over the recall they reach.')
    sup_record('s5_rq2_operating_points', dict(targets=int(len(IDX_DONE)), rows=OP.to_dict('records')))
    fig, axes = plt.subplots(2, 2, figsize=(10, 7.4), constrained_layout=True)
    markers = ['o', 's', '^', 'D', 'v']
    for r_, level in enumerate(['physical', 'group']):
        axr, axp = axes[r_]
        for (lv, lab), c in curves.items():
            if lv != level: continue
            ls = '-' if 'full' in lab else '--'
            axr.plot(np.maximum(c['fpr'], 1e-7), c['tpr'], ls=ls, lw=1.3, label=lab); axp.plot(c['recall'], c['precision'], ls=ls, lw=1.3, label=lab)
        pp_, nn_ = rq_weights(level); prev = pp_[cm].sum() / (pp_[cm].sum() + nn_[cm].sum())
        for k_, (name, A_) in enumerate(nets.items()):
            d = OP[(OP.level == level) & (OP.method == name)].iloc[0]
            axr.plot(max(d.FPR, 1e-7), d.TPR, marker=markers[k_ % 5], ms=7, ls='none', mec='k', label=f'{name} ({int(d.edges)} edges)')
            # a network with zero true positives has precision 0, which a log axis cannot show: draw it at the floor
            axp.plot(max(d.recall, 1e-4), max(d.precision, prev / 20), marker=markers[k_ % 5], ms=7, ls='none', mec='k', mfc='none' if d.precision == 0 else None)
        axp.axhline(prev, c='grey', ls=':', lw=1); axp.text(.98, prev, 'prevalence', ha='right', va='bottom', fontsize=7, color='grey')
        axr.plot([1e-7, 1], [1e-7, 1], c='grey', ls=':', lw=1)
        axr.set(xscale='log', xlabel='False-positive rate (log)', ylabel='True-positive rate', title=f'ROC, {level} level', xlim=(1e-7, 1), ylim=(0, 1))
        axp.set(xscale='log', yscale='log', xlabel='Recall (log)', ylabel='Precision (log)', title=f'Precision-recall, {level} level')
    axes[0, 0].legend(fontsize=6.5, loc='upper left')
    sup_fig(fig, 'figS5_rq2_roc_pr', 'RQ2: pairwise lag-1 TE ranking (ROC and precision-recall) on the analysed targets with the four networks as operating points and the IDTxl partial-ranking curves; log axes expose the sparse regime.')

## Index of supplementary outputs

In [ ]:
(SUP_DIR / 'figure_index.md').write_text('\n'.join(f'- `{k}.pdf`: {v}' for k, v in FIGURES.items()))
(SUP_DIR / 'table_index.md').write_text('\n'.join(f'- `{k}.tex`: {v}' for k, v in TABLES.items()))
sup_record('provenance', dict(run_directory=str(OUT), results_directory=str(RQ_DIR), toolkit_version=tk.TOOLKIT_VERSION,
                              finished=time.strftime('%Y-%m-%d %H:%M:%S'), figures=list(FIGURES), tables=list(TABLES)))
print('Figures:', len(FIGURES), '| Tables:', len(TABLES), '| written to', SUP_DIR)
display(Markdown('\n'.join([f'- **{k}** - {v}' for k, v in FIGURES.items()] + [f'- `{k}.tex` - {v}' for k, v in TABLES.items()])))